In [ ]:
# Required Libraries
import os
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Environment loaded")

## 1. Configuration

In [ ]:
# Azure OpenAI credentials
AZURE_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT", "https://your-endpoint.cognitiveservices.azure.com/")
DEPLOYMENT_NAME = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME", "gpt-4.1")
API_VERSION = os.getenv("AI_FOUNDRY_API_VERSION", "2024-12-01-preview")
API_KEY = os.getenv("AI_FOUNDRY_API_KEY", "")

print("🔧 Configuration:")
print(f"   Endpoint: {AZURE_ENDPOINT}")
print(f"   Deployment: {DEPLOYMENT_NAME}")
print(f"   API Key: {'Set ✅' if API_KEY else 'NOT SET ❌'}")

## 2. Create Model Client

In [ ]:
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

model_client = AzureOpenAIChatCompletionClient(
    azure_endpoint=AZURE_ENDPOINT,
    model=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=API_KEY,
)

print("✅ Model client created")

## 3. Create Specialized ML Agents

In [ ]:
from autogen_agentchat.agents import AssistantAgent

# Agent 1: Data Scientist
data_scientist = AssistantAgent(
    name="DataScientist",
    model_client=model_client,
    system_message="""
    You are the Lead Data Scientist.
    
    Your responsibilities:
    1. Analyze the dataset and problem type (classification/regression)
    2. Identify key features and their importance
    3. Recommend data preprocessing steps
    4. Suggest appropriate ML algorithms
    
    After your analysis, the ModelBuilder will implement the training code.
    Be specific about:
    - Feature engineering recommendations
    - Model selection rationale
    - Expected challenges
    """
)

# Agent 2: Model Builder
model_builder = AssistantAgent(
    name="ModelBuilder",
    model_client=model_client,
    system_message="""
    You are the ML Model Builder.
    
    Your responsibilities:
    1. Write complete, working Python code for ML training
    2. Implement data preprocessing (imputation, encoding, scaling)
    3. Train multiple baseline models
    4. Include proper train/test split
    
    Use these libraries: pandas, numpy, sklearn
    
    Your code should:
    - Be complete and runnable
    - Include comments
    - Handle edge cases
    - Print model results
    """
)

# Agent 3: Evaluator
evaluator = AssistantAgent(
    name="Evaluator",
    model_client=model_client,
    system_message="""
    You are the Model Evaluator.
    
    Your responsibilities:
    1. Write code for comprehensive model evaluation
    2. Generate classification metrics (accuracy, precision, recall, F1, confusion matrix)
    3. Generate regression metrics (RMSE, MAE, R2)
    4. Implement cross-validation
    
    Your evaluation code should:
    - Use the model from ModelBuilder's code
    - Create visualizations (confusion matrix, feature importance)
    - Provide clear interpretation of results
    """
)

# Agent 4: Hyperparameter Tuner
tuner = AssistantAgent(
    name="HyperparameterTuner",
    model_client=model_client,
    system_message="""
    You are the Hyperparameter Tuning Expert.
    
    Your responsibilities:
    1. Write code for hyperparameter optimization
    2. Use Optuna for Bayesian optimization
    3. Define appropriate search spaces
    4. Report best parameters and improved scores
    
    Your tuning code should:
    - Build on the ModelBuilder's baseline
    - Use Optuna with TPE sampler
    - Run 20-50 trials
    - Compare tuned vs baseline performance
    """
)

print("✅ Created 4 specialized agents:")
print(f"   1. {data_scientist.name}")
print(f"   2. {model_builder.name}")
print(f"   3. {evaluator.name}")
print(f"   4. {tuner.name}")

## 4. Create Multi-Agent Team

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

# Create a team with round-robin communication
# Each agent takes a turn in order
ml_team = RoundRobinGroupChat(
    participants=[data_scientist, model_builder, evaluator, tuner],
    max_turns=8,  # 2 rounds of all 4 agents
)

print("✅ Multi-agent team created")
print(f"   Max turns: 8 (2 rounds × 4 agents)")

## 5. Run Multi-Agent Pipeline

In [ ]:
async def run_ml_pipeline(team, task: str):
    """
    Run the multi-agent ML pipeline.
    Collects all messages from the team conversation.
    """
    print("="*60)
    print("🚀 STARTING MULTI-AGENT ML PIPELINE")
    print("="*60)
    print(f"\n📋 Task: {task[:100]}...")
    print("\n" + "-"*60)
    
    messages = []
    
    # Run team with streaming
    async for event in team.run_stream(task=task):
        # Handle different event types
        if hasattr(event, 'source') and hasattr(event, 'content'):
            source = event.source
            content = event.content
            
            messages.append({
                'agent': source,
                'content': content
            })
            
            # Print agent response (truncated for display)
            print(f"\n🤖 [{source}]:")
            if len(content) > 500:
                print(content[:500] + "...")
                print(f"\n   [Response truncated - full length: {len(content)} chars]")
            else:
                print(content)
            print("-"*60)
    
    print("\n" + "="*60)
    print(f"✅ PIPELINE COMPLETE - {len(messages)} messages")
    print("="*60)
    
    return messages

In [ ]:
# Define the ML task
ml_task = """
Build a machine learning model for the Titanic dataset.

Dataset: titanic.csv
Target column: Survived
Problem type: Binary Classification

Columns: PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked

Requirements:
1. DataScientist: Analyze the problem and recommend approach
2. ModelBuilder: Write complete training code with preprocessing
3. Evaluator: Write evaluation code with metrics and visualizations
4. HyperparameterTuner: Write Optuna tuning code

Each agent should provide working Python code.
"""

# Run the pipeline
results = await run_ml_pipeline(ml_team, ml_task)

## 6. Process Results

In [ ]:
# Summarize agent contributions
print("\n📊 AGENT CONTRIBUTIONS SUMMARY")
print("="*60)

for i, msg in enumerate(results):
    agent = msg['agent']
    content = msg['content']
    
    # Count code blocks
    code_blocks = content.count('```python')
    
    print(f"\n{i+1}. {agent}:")
    print(f"   - Response length: {len(content)} characters")
    print(f"   - Code blocks: {code_blocks}")

In [ ]:
# Extract code from a specific agent
import re

def extract_code(content: str) -> list:
    """Extract Python code blocks from agent response."""
    pattern = r'```python\n(.*?)```'
    matches = re.findall(pattern, content, re.DOTALL)
    return matches

# Get ModelBuilder's code
for msg in results:
    if msg['agent'] == 'ModelBuilder':
        code_blocks = extract_code(msg['content'])
        if code_blocks:
            print("📝 ModelBuilder's Code:")
            print("="*60)
            print(code_blocks[0][:1000])  # First 1000 chars
            if len(code_blocks[0]) > 1000:
                print("...")
        break

## 7. Alternative: Selector Group Chat

In [ ]:
from autogen_agentchat.teams import SelectorGroupChat

# SelectorGroupChat uses the model to decide which agent speaks next
# This is more dynamic than round-robin

selector_team = SelectorGroupChat(
    participants=[data_scientist, model_builder, evaluator, tuner],
    model_client=model_client,
    max_turns=8,
)

print("✅ SelectorGroupChat team created")
print("   This team uses AI to select which agent responds next")

## 8. Save Results to File

In [ ]:
import json
from datetime import datetime

# Save conversation to JSON
output = {
    'timestamp': datetime.now().isoformat(),
    'task': ml_task,
    'agents': [data_scientist.name, model_builder.name, evaluator.name, tuner.name],
    'messages': results
}

output_file = f"../artifacts/multi_agent_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f"✅ Results saved to: {output_file}")

## ✅ Summary

This module demonstrates:

**1. Multi-Agent Setup**
- 4 specialized agents: DataScientist, ModelBuilder, Evaluator, HyperparameterTuner
- Each with focused system messages

**2. Team Orchestration**
- `RoundRobinGroupChat`: Fixed turn order
- `SelectorGroupChat`: AI-driven agent selection

**3. Running Pipelines**
- `team.run_stream()` for async streaming
- Collect all agent messages
- Process and extract code

**4. Key Autogen 0.4+ Imports**
```python
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
```

**Best Practices:**
- Clear, focused system messages
- Limit max_turns to control costs
- Use streaming for real-time output
- Save results for reproducibility